# ArcNeuron on Google Colab

Notebook này chỉ là runner mỏng cho repo. Nó không chứa kiến trúc, luật suy luận, knowledge base hay training logic riêng.
`arcneuron.py`, `train.py`, `tune.py`, `tokenizer.py` và `generate.py` vẫn là source of truth.

Bật GPU trong **Runtime → Change runtime type → GPU** trước khi chạy.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO = "https://github.com/ArcatureLabs/ArcNeuron.git"
ROOT = Path("/content/ArcNeuron")

if not (ROOT / "arcneuron.py").is_file():
    if ROOT.exists():
        subprocess.run(["rm", "-rf", str(ROOT)], check=True)
    subprocess.run(["git", "clone", "--depth", "1", REPO, str(ROOT)], check=True)

os.chdir(ROOT)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "sentencepiece"], check=True)
print("working directory:", Path.cwd())


In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError("GPU chưa được bật. Chọn Runtime > Change runtime type > GPU rồi chạy lại.")

print("PyTorch:", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0))
print("BF16:", torch.cuda.is_bf16_supported())
props = torch.cuda.get_device_properties(0)
print(f"VRAM: {props.total_memory / 1024**3:.1f} GiB")


## Cấu hình thử nghiệm

Mặc định cố tình nhỏ để kiểm tra pipeline nhanh. Sau khi chạy ổn mới tăng kích thước và số bước.


In [ ]:
TRAIN_STEPS = 300
TUNE_STEPS = 100
BATCH_SIZE = 8
CONTEXT = 256

DIM = 256
HEADS = 4
KV_HEADS = 1
FFN_DIM = 704
PRELUDE_LAYERS = 1
CORE_LAYERS = 2
CODA_LAYERS = 1
MAX_DEPTH = 4

BASE_CKPT = "arcneuron.pt"
TUNED_CKPT = "arcneuron-tuned.pt"


## Train base model

Cell này chỉ gọi `train.py`; notebook không sao chép training loop hay reasoning logic.


In [ ]:
cmd = [
    sys.executable, "train.py",
    "--data", "train.txt",
    "--out", BASE_CKPT,
    "--steps", str(TRAIN_STEPS),
    "--batch-size", str(BATCH_SIZE),
    "--context", str(CONTEXT),
    "--dim", str(DIM),
    "--heads", str(HEADS),
    "--kv-heads", str(KV_HEADS),
    "--ffn-dim", str(FFN_DIM),
    "--prelude-layers", str(PRELUDE_LAYERS),
    "--core-layers", str(CORE_LAYERS),
    "--coda-layers", str(CODA_LAYERS),
    "--max-depth", str(MAX_DEPTH),
    "--eval-every", "50",
    "--eval-batches", "4",
]
subprocess.run(cmd, check=True)


## Tuning

Tuning tiếp tục cập nhật trực tiếp cùng weights bằng next-token training, không adapter hay model phụ.


In [ ]:
cmd = [
    sys.executable, "tune.py",
    "--checkpoint", BASE_CKPT,
    "--data", "tune.txt",
    "--replay-data", "train.txt",
    "--out", TUNED_CKPT,
    "--steps", str(TUNE_STEPS),
    "--batch-size", str(max(1, BATCH_SIZE // 2)),
    "--context", str(min(CONTEXT, 256)),
    "--max-depth", str(MAX_DEPTH),
    "--save-every", str(TUNE_STEPS),
]
subprocess.run(cmd, check=True)


## Generate

`DEPTH` là số lần dùng lại recurrent core và là test-time compute knob trực tiếp của ArcNeuron.


In [ ]:
from generate import load_model, generate

device = torch.device("cuda")
checkpoint = TUNED_CKPT if Path(TUNED_CKPT).is_file() else BASE_CKPT
model, tokenizer = load_model(checkpoint, device)

PROMPT = "Một con mèo bị mất một chân có còn là động vật có vú không? Giải thích."
DEPTH = 4

text = generate(
    model=model,
    tokenizer=tokenizer,
    prompt=PROMPT,
    depth=DEPTH,
    max_new_tokens=160,
    temperature=0.8,
    top_k=50,
    device=device,
)
print(text)


## So sánh recurrent depth

Cùng checkpoint và prompt, chỉ đổi số vòng để xem thêm compute có giúp thật hay gây overthinking.


In [ ]:
for depth in [1, 2, 4, 8]:
    torch.manual_seed(42)
    torch.cuda.manual_seed_all(42)
    text = generate(
        model=model,
        tokenizer=tokenizer,
        prompt=PROMPT,
        depth=depth,
        max_new_tokens=160,
        temperature=0.0,
        top_k=0,
        device=device,
    )
    print(f"\n{'=' * 24} depth={depth} {'=' * 24}\n")
    print(text)


## Tải checkpoint khỏi Colab

Chạy cell này trước khi runtime reset nếu muốn giữ checkpoint ở máy cá nhân.


In [ ]:
from google.colab import files
path = TUNED_CKPT if Path(TUNED_CKPT).is_file() else BASE_CKPT
files.download(path)
